# Phase 4 — Enhanced Streamlit Dashboard

## New features added
- PDF upload for resume and job description
- Baseline model comparison (Keyword Overlap + TF-IDF vs our model)
- Full SHAP reasoning with per-feature impact explanation
- Updated resume generator (ATS optimized)
- No emojis in UI

---
Upload all output files before running Cell 3.

## Cell 1 — Install packages

In [ ]:
!pip install -q streamlit pyngrok sentence-transformers faiss-cpu xgboost shap PyPDF2 scikit-learn
print('Done!')

## Cell 2 — Write app.py to disk

In [ ]:
app_code = "import streamlit as st\nimport json, pickle, re, warnings, io, textwrap\nimport numpy as np\nimport pandas as pd\nimport matplotlib.pyplot as plt\nimport matplotlib.gridspec as gridspec\nimport faiss\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.metrics.pairwise import cosine_similarity\nfrom sentence_transformers import SentenceTransformer\nimport PyPDF2\nwarnings.filterwarnings('ignore')\n\nst.set_page_config(page_title='AI Resume Matcher', layout='wide')\n\n# \u2500\u2500 CSS \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\nst.markdown(\"\"\"\n<style>\n.big-score{font-size:68px;font-weight:800;text-align:center;line-height:1}\n.sub-score{text-align:center;color:gray;font-size:15px}\n.badge{padding:6px 22px;border-radius:20px;font-size:20px;font-weight:700;display:inline-block}\n.shortlist{background:#d4edda;color:#155724}\n.maybe{background:#fff3cd;color:#856404}\n.reject{background:#f8d7da;color:#721c24}\n.stag{display:inline-block;padding:4px 10px;border-radius:10px;font-size:13px;margin:2px}\n.section-header{font-size:17px;font-weight:700;margin-top:18px;margin-bottom:6px;border-bottom:2px solid #eee;padding-bottom:4px}\n</style>\n\"\"\", unsafe_allow_html=True)\n\n# \u2500\u2500 Skill & education constants \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\nKNOWN_SKILLS = {\n    'python','java','sql','r','scala','c++','javascript','typescript',\n    'machine learning','deep learning','nlp','computer vision',\n    'tensorflow','pytorch','keras','scikit-learn','xgboost','lightgbm',\n    'pandas','numpy','matplotlib','seaborn','plotly',\n    'spark','hadoop','kafka','airflow','dbt',\n    'aws','azure','gcp','docker','kubernetes','mlflow',\n    'mysql','postgresql','mongodb','redis','elasticsearch',\n    'git','linux','rest api','flask','fastapi','django',\n    'tableau','power bi','excel','statistics','probability',\n    'data analysis','data visualization','feature engineering',\n    'model deployment','ci/cd','agile','communication'\n}\nCRITICAL_SKILLS = {\n    'python','sql','machine learning','deep learning','statistics',\n    'data analysis','aws','docker','spark','tensorflow','pytorch'\n}\nEDU_RANK = {\n    'phd':5,'doctorate':5,'m.tech':4,'msc':4,'mba':4,'master':4,\n    'b.tech':3,'bsc':3,'bachelor':3,'be':3,'diploma':2\n}\n\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n# HELPERS\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\ndef extract_text_from_pdf(uploaded_file) -> str:\n    reader = PyPDF2.PdfReader(io.BytesIO(uploaded_file.read()))\n    text = ''\n    for page in reader.pages:\n        text += (page.extract_text() or '') + '\\n'\n    return text.strip()\n\ndef extract_skills(text):\n    tl = text.lower()\n    found = [s for s in KNOWN_SKILLS if s in tl]\n    m = re.search(r'skills?[\\:\\s]+([^\\n]+)', tl)\n    if m:\n        found.extend([s.strip() for s in re.split(r'[,|;]', m.group(1)) if len(s.strip()) > 2])\n    return list(set(found))\n\ndef extract_exp(text):\n    for p in [r'(\\d+\\.?\\d*)\\s*\\+?\\s*years?\\s+of\\s+experience',\n              r'minimum\\s+(\\d+)\\s+years?',\n              r'(\\d+\\.?\\d*)\\s*\\+?\\s*years?\\s+experience']:\n        m = re.search(p, text.lower())\n        if m: return float(m.group(1))\n    nums = re.findall(r'(\\d+\\.?\\d*)\\s*years?', text.lower())\n    return sum(float(e) for e in nums[:3]) if nums else 0.0\n\ndef extract_edu(text):\n    t = text.lower()\n    for deg, rank in sorted(EDU_RANK.items(), key=lambda x: -x[1]):\n        if deg in t: return deg, rank\n    return 'bachelor', 3\n\ndef normalize_skills(skills, sbert, fi, corpus):\n    if not skills: return []\n    emb = sbert.encode([s.lower() for s in skills],\n                       convert_to_numpy=True, normalize_embeddings=True).astype('float32')\n    dists, idxs = fi.search(emb, k=1)\n    return list(set([\n        corpus[idxs[i][0]] if dists[i][0] > 0.75 else skills[i].lower()\n        for i in range(len(skills))\n    ]))\n\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n# BASELINE MODELS (for comparison)\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\ndef baseline_keyword_overlap(resume_text, jd_text):\n    r_words = set(re.findall(r'\\b\\w+\\b', resume_text.lower()))\n    j_words = set(re.findall(r'\\b\\w+\\b', jd_text.lower()))\n    stopwords = {'the','a','an','is','in','of','to','and','or','for','with','on','at','by','from','as','are','was','were','be','been','being','have','has','had','do','does','did','will','would','could','should','may','might','this','that','these','those','i','you','he','she','it','we','they'}\n    r_words -= stopwords; j_words -= stopwords\n    overlap = r_words & j_words\n    score = round(len(overlap) / len(j_words) * 100, 1) if j_words else 0\n    return score, list(overlap)[:10]\n\ndef baseline_tfidf(resume_text, jd_text):\n    vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1,2))\n    try:\n        tfidf_matrix = vectorizer.fit_transform([resume_text, jd_text])\n        sim = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])[0][0]\n        return round(sim * 100, 1)\n    except:\n        return 0.0\n\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n# UPDATED RESUME GENERATOR\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\ndef generate_updated_resume(resume_text, parsed_resume, missing_skills, jd_title, improvements):\n    lines = []\n    lines.append(\"=\" * 60)\n    lines.append(\"UPDATED RESUME  (Optimized for ATS)\")\n    lines.append(\"=\" * 60)\n\n    name_m = re.search(r'^([A-Z][a-z]+ [A-Z][a-z]+)', resume_text.strip())\n    email_m = re.search(r'[\\w.+-]+@[\\w-]+\\.[\\w.]+', resume_text)\n    name  = name_m.group(1) if name_m else 'Candidate'\n    email = email_m.group(0) if email_m else ''\n\n    lines.append(f\"\\n{name}\")\n    if email: lines.append(email)\n    lines.append(\"\")\n\n    lines.append(\"CAREER OBJECTIVE\")\n    lines.append(\"-\" * 40)\n    lines.append(\n        f\"Results-driven professional seeking a {jd_title} role. \"\n        f\"Experienced in {', '.join(parsed_resume.get('skills', [])[:4])}. \"\n        f\"Eager to leverage expertise in a challenging environment.\"\n    )\n    lines.append(\"\")\n\n    lines.append(\"SKILLS\")\n    lines.append(\"-\" * 40)\n    current_skills = parsed_resume.get('skills', [])\n    must_add = [s['skill'] for s in missing_skills if s['importance'] == 'Must-have']\n    good_add = [s['skill'] for s in missing_skills if s['importance'] == 'Good-to-have']\n    all_skills = list(set(current_skills + must_add))\n    lines.append(', '.join([s.title() for s in all_skills]))\n    if must_add:\n        lines.append(f\"\\n[ATS NOTE: Added critical missing skills: {', '.join(must_add)}]\")\n    if good_add:\n        lines.append(f\"[ATS NOTE: Consider adding: {', '.join(good_add[:3])}]\")\n    lines.append(\"\")\n\n    exp_section = re.search(\n        r'(experience|work history)(.*?)(education|projects|skills|$)',\n        resume_text, re.I | re.S\n    )\n    if exp_section:\n        lines.append(\"EXPERIENCE\")\n        lines.append(\"-\" * 40)\n        exp_text = exp_section.group(2).strip()[:800]\n        lines.append(exp_text)\n        lines.append(\"\")\n        lines.append(\"[ATS NOTE: Add measurable outcomes to each bullet point,\")\n        lines.append(\" e.g. 'Improved model accuracy by 15% using XGBoost']\")\n        lines.append(\"\")\n\n    edu_section = re.search(\n        r'(education)(.*?)(experience|projects|skills|certifications|$)',\n        resume_text, re.I | re.S\n    )\n    if edu_section:\n        lines.append(\"EDUCATION\")\n        lines.append(\"-\" * 40)\n        lines.append(edu_section.group(2).strip()[:400])\n        lines.append(\"\")\n\n    lines.append(\"IMPROVEMENT CHECKLIST\")\n    lines.append(\"-\" * 40)\n    for i, tip in enumerate(improvements, 1):\n        lines.append(f\"{i}. {tip}\")\n\n    lines.append(\"\")\n    lines.append(\"=\" * 60)\n    lines.append(\"Generated by AI Resume Matcher\")\n    lines.append(\"=\" * 60)\n\n    return '\\n'.join(lines)\n\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n# LOAD MODELS\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n@st.cache_resource\ndef load_models():\n    with open('/content/preprocessing_config.json') as f:\n        config = json.load(f)\n    feature_cols = config['feature_cols']\n    with open('/content/XGB Regressor Model.pkl','rb') as f:   reg = pickle.load(f)\n    with open('/content/XGB Classifier Model.pkl','rb') as f:  cls = pickle.load(f)\n    with open('/content/Label Encoder Model.pkl','rb') as f:   le  = pickle.load(f)\n    with open('/content/Shap Explainer Model Training.pkl','rb') as f: shap_exp = pickle.load(f)\n    sbert = SentenceTransformer('all-MiniLM-L6-v2')\n    fi    = faiss.read_index('/content/skill_faiss.index')\n    with open('/content/skill_corpus.json') as f: corpus = json.load(f)\n    return feature_cols, reg, cls, le, shap_exp, sbert, fi, corpus\n\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n# MAIN PIPELINE\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\ndef run_pipeline(resume_text, jd_text, feature_cols, reg, cls, le, shap_exp, sbert, fi, corpus):\n    name_m  = re.search(r'^([A-Z][a-z]+ [A-Z][a-z]+)', resume_text.strip())\n    title_m = re.search(r'(?:role|job title|position)[\\:\\s]+([^\\n]+)', jd_text, re.I)\n    r_skills = extract_skills(resume_text); r_exp = extract_exp(resume_text)\n    r_deg, r_edu = extract_edu(resume_text); r_len = len(resume_text.split())\n    j_skills = extract_skills(jd_text);     j_exp = extract_exp(jd_text)\n    j_deg, j_edu = extract_edu(jd_text)\n    candidate = name_m.group(1)  if name_m  else 'Candidate'\n    job_title = title_m.group(1).strip() if title_m else 'Target Role'\n\n    norm_r = normalize_skills(r_skills, sbert, fi, corpus)\n    norm_j = normalize_skills(j_skills, sbert, fi, corpus)\n    r_set, j_set = set(norm_r), set(norm_j)\n    overlap = r_set & j_set\n    skill_overlap = len(overlap) / len(j_set) if j_set else 0.0\n\n    emb_r = sbert.encode([' '.join(norm_r)], normalize_embeddings=True)\n    emb_j = sbert.encode([' '.join(norm_j)], normalize_embeddings=True)\n    sbert_sim = float(np.dot(emb_r, emb_j.T)[0][0])\n    exp_gap   = max(0.0, j_exp - r_exp)\n    edu_match = 1 if r_edu >= j_edu else 0\n\n    fm = {\n        'feat_skill_overlap':       skill_overlap,\n        'feat_resume_skill_count':  len(norm_r),\n        'feat_jd_skill_count':      len(norm_j),\n        'feat_missing_skill_count': len(j_set - r_set),\n        'feat_edu_match':           edu_match,\n        'feat_edu_level_resume':    r_edu,\n        'feat_edu_level_jd':        j_edu,\n        'feat_exp_years_required':  j_exp,\n        'feat_resume_text_len':     r_len,\n        'feat_sbert_similarity':    sbert_sim,\n        'feat_resume_exp_years':    r_exp,\n        'feat_exp_gap':             exp_gap\n    }\n    X = pd.DataFrame([[fm.get(c, 0) for c in feature_cols]], columns=feature_cols)\n    match_score = float(np.clip(reg.predict(X)[0], 0, 1))\n    label       = le.inverse_transform([cls.predict(X)[0]])[0]\n    skill_score = round(skill_overlap * 100, 1)\n    sbert_score = round(sbert_sim * 100, 1)\n    exp_score   = round(max(0, 1 - exp_gap / max(j_exp, 1)) * 100, 1)\n\n    missing_skills = [\n        {'skill': s, 'importance': 'Must-have' if s in CRITICAL_SKILLS else 'Good-to-have'}\n        for s in (j_set - r_set)\n    ]\n    missing_skills.sort(key=lambda x: 0 if x['importance'] == 'Must-have' else 1)\n\n    shap_vals = shap_exp.shap_values(X)\n    top_pos = sorted(zip(feature_cols, shap_vals[0]), key=lambda x: x[1], reverse=True)\n    top_neg = sorted(zip(feature_cols, shap_vals[0]), key=lambda x: x[1])\n    pos_factors = [f.replace('feat_','').replace('_',' ') for f,v in top_pos[:3] if v > 0]\n    neg_factors = [f.replace('feat_','').replace('_',' ') for f,v in top_neg[:3] if v < 0]\n    missing_must = [s['skill'] for s in missing_skills if s['importance'] == 'Must-have']\n\n    if label == 'Shortlist': dec = f'a strong candidate for {job_title}'\n    elif label == 'Maybe':   dec = f'a borderline match for {job_title}'\n    else:                    dec = f'not yet ready for {job_title}'\n\n    explanation = (\n        f\"{candidate} scored {round(match_score*100,1)}/100 and is {dec}. \"\n        f\"Matching skills: {', '.join(list(overlap)[:5]) or 'none'}. \"\n    )\n    if missing_must: explanation += f\"Critical gaps: {', '.join(missing_must)}. \"\n    if exp_gap > 0:  explanation += f\"Experience is {exp_gap:.1f} years below requirement.\"\n\n    improvements = []\n    if missing_must: improvements.append(f\"Add missing critical skills to resume: {', '.join(missing_must[:3])}.\")\n    if exp_gap > 0:  improvements.append(f\"Bridge {exp_gap:.1f}-year experience gap with projects or freelance work.\")\n    if sbert_score < 60: improvements.append(\"Mirror JD keywords exactly in your resume bullets.\")\n    while len(improvements) < 3:\n        improvements.append(\"Add quantifiable achievements (e.g. improved model accuracy by 15%).\")\n\n    ats_score = min(100, int(skill_score*0.5 + sbert_score*0.3 + (100 if exp_gap==0 else 50)*0.2))\n\n    kw_score, kw_overlap = baseline_keyword_overlap(resume_text, jd_text)\n    tfidf_score = baseline_tfidf(resume_text, jd_text)\n\n    parsed_resume = {'skills': list(r_set), 'exp': r_exp, 'edu': r_deg}\n    updated_resume = generate_updated_resume(\n        resume_text, parsed_resume, missing_skills, job_title, improvements\n    )\n\n    return {\n        'candidate': candidate, 'job_title': job_title,\n        'overall_score': round(match_score * 100, 1), 'label': label,\n        'section_scores': {\n            'Skill Match':  skill_score,\n            'Semantic Fit': sbert_score,\n            'Experience':   exp_score,\n            'Role Fit':     round(match_score * 100, 1)\n        },\n        'matched_skills': list(overlap), 'missing_skills': missing_skills,\n        'explanation': explanation, 'improvements': improvements[:3],\n        'ats_score': ats_score,\n        'pos_factors': pos_factors, 'neg_factors': neg_factors,\n        'shap_vals': shap_vals[0].tolist(), 'feature_cols': feature_cols,\n        'baseline_keyword': kw_score,\n        'baseline_tfidf':   tfidf_score,\n        'kw_overlap_words': kw_overlap,\n        'updated_resume':   updated_resume\n    }\n\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n# UI\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\nst.title(\"AI Resume - Job Description Matcher\")\nst.markdown(\"*Semantic matching with explainability \u2014 SBERT + XGBoost + SHAP*\")\nst.divider()\n\nwith st.spinner(\"Loading models...\"):\n    try:\n        feature_cols, reg, cls, le, shap_exp, sbert, fi, corpus = load_models()\n        st.success(\"Models loaded successfully.\")\n    except Exception as e:\n        st.error(f\"Could not load models: {e}\")\n        st.stop()\n\n# \u2500\u2500 Input section \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\ncol1, col2 = st.columns(2)\n\nwith col1:\n    st.subheader(\"Resume\")\n    resume_input_type = st.radio(\"Input type\", [\"Upload PDF\", \"Paste Text\"], key=\"r_type\", horizontal=True)\n    if resume_input_type == \"Upload PDF\":\n        resume_file = st.file_uploader(\"Upload resume PDF\", type=[\"pdf\"], key=\"r_pdf\")\n        resume_text = extract_text_from_pdf(resume_file) if resume_file else \"\"\n        if resume_text:\n            st.caption(f\"Extracted {len(resume_text.split())} words from PDF\")\n    else:\n        resume_text = st.text_area(\"Paste resume text\", height=260,\n                                   placeholder=\"John Doe\\nSkills: Python, SQL...\")\n\nwith col2:\n    st.subheader(\"Job Description\")\n    jd_input_type = st.radio(\"Input type\", [\"Upload PDF\", \"Paste Text\"], key=\"j_type\", horizontal=True)\n    if jd_input_type == \"Upload PDF\":\n        jd_file = st.file_uploader(\"Upload JD PDF\", type=[\"pdf\"], key=\"j_pdf\")\n        jd_text = extract_text_from_pdf(jd_file) if jd_file else \"\"\n        if jd_text:\n            st.caption(f\"Extracted {len(jd_text.split())} words from PDF\")\n    else:\n        jd_text = st.text_area(\"Paste job description text\", height=260,\n                               placeholder=\"Role: Data Scientist\\nRequired Skills: Python...\")\n\nst.divider()\nrun_btn = st.button(\"Analyze Match\", type=\"primary\", use_container_width=True)\n\n# \u2500\u2500 Results \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\nif run_btn:\n    if not resume_text.strip() or not jd_text.strip():\n        st.warning(\"Please provide both a resume and a job description.\")\n    else:\n        with st.spinner(\"Running 5-agent pipeline...\"):\n            result = run_pipeline(resume_text, jd_text,\n                                  feature_cols, reg, cls, le, shap_exp, sbert, fi, corpus)\n\n        st.divider()\n\n        # \u2500\u2500 Summary row \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n        c1, c2, c3 = st.columns([1, 1, 2])\n        score = result['overall_score']\n        color = '#28a745' if score >= 75 else '#ffc107' if score >= 50 else '#dc3545'\n        with c1:\n            st.markdown(\n                f\"<div class='big-score' style='color:{color}'>{score}</div>\"\n                f\"<div class='sub-score'>/100  Overall Score</div>\",\n                unsafe_allow_html=True\n            )\n        with c2:\n            badge_color = {'Shortlist':'#d4edda','Maybe':'#fff3cd','Reject':'#f8d7da'}[result['label']]\n            text_color  = {'Shortlist':'#155724','Maybe':'#856404','Reject':'#721c24'}[result['label']]\n            st.markdown(\n                f\"<div class='badge' style='background:{badge_color};color:{text_color}'>\"\n                f\"{result['label']}</div>\",\n                unsafe_allow_html=True\n            )\n            st.markdown(f\"**ATS Readiness:** {result['ats_score']}/100\")\n        with c3:\n            st.markdown(f\"**Candidate:** {result['candidate']}\")\n            st.markdown(f\"**Role:** {result['job_title']}\")\n            st.markdown(\"**Key Strengths:** \" + (\", \".join(result['pos_factors']) or \"\u2014\"))\n            st.markdown(\"**Weak Areas:** \"    + (\", \".join(result['neg_factors']) or \"\u2014\"))\n\n        st.divider()\n\n        # \u2500\u2500 Section scores \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n        st.markdown(\"<div class='section-header'>Section-wise Scores</div>\", unsafe_allow_html=True)\n        scores = result['section_scores']\n        fig, ax = plt.subplots(figsize=(10, 3))\n        colors2 = ['#28a745' if v >= 75 else '#ffc107' if v >= 50 else '#dc3545' for v in scores.values()]\n        bars = ax.barh(list(scores.keys()), list(scores.values()), color=colors2, edgecolor='none', height=0.5)\n        ax.set_xlim(0, 100)\n        ax.axvline(75, color='green',  linestyle='--', linewidth=1, alpha=0.5, label='Good (75)')\n        ax.axvline(50, color='orange', linestyle='--', linewidth=1, alpha=0.5, label='Fair (50)')\n        for bar, val in zip(bars, scores.values()):\n            ax.text(bar.get_width()+1, bar.get_y()+bar.get_height()/2,\n                    f'{val}/100', va='center', fontsize=11)\n        ax.legend(fontsize=9); ax.spines[['top','right']].set_visible(False)\n        plt.tight_layout(); st.pyplot(fig); plt.close()\n\n        st.divider()\n\n        # \u2500\u2500 Skills \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n        c1, c2 = st.columns(2)\n        with c1:\n            st.markdown(\"<div class='section-header'>Matched Skills</div>\", unsafe_allow_html=True)\n            if result['matched_skills']:\n                st.markdown(\n                    ''.join([f\"<span class='stag' style='background:#d4edda;color:#155724'>{s}</span>\"\n                             for s in result['matched_skills']]),\n                    unsafe_allow_html=True\n                )\n            else:\n                st.info(\"No exact skill matches found.\")\n        with c2:\n            st.markdown(\"<div class='section-header'>Missing Skills</div>\", unsafe_allow_html=True)\n            if result['missing_skills']:\n                st.markdown(\n                    ''.join([\n                        f\"<span class='stag' style='background:\"\n                        f\"{'#f8d7da' if s['importance']=='Must-have' else '#fff3cd'};\"\n                        f\"color:{'#721c24' if s['importance']=='Must-have' else '#856404'}'>\"\n                        f\"{s['skill']} ({s['importance']})</span>\"\n                        for s in result['missing_skills']\n                    ]),\n                    unsafe_allow_html=True\n                )\n            else:\n                st.success(\"No skill gaps found.\")\n\n        st.divider()\n\n        # \u2500\u2500 SHAP explainability \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n        st.markdown(\"<div class='section-header'>Model Reasoning \u2014 SHAP Explainability</div>\", unsafe_allow_html=True)\n        st.caption(\n            \"SHAP (SHapley Additive exPlanations) shows exactly which features pushed \"\n            \"the score up (green) or down (red) for this specific resume-JD pair. \"\n            \"Longer bars = stronger impact.\"\n        )\n        sv = result['shap_vals']\n        fn = [f.replace('feat_','').replace('_',' ').title() for f in result['feature_cols']]\n        sp = sorted(zip(fn, sv), key=lambda x: abs(x[1]), reverse=True)[:8]\n        ns, vs = zip(*sp)\n\n        fig2, ax2 = plt.subplots(figsize=(10, 4))\n        bar_colors = ['#28a745' if v > 0 else '#dc3545' for v in vs]\n        bars2 = ax2.barh(ns, vs, color=bar_colors, edgecolor='none', height=0.5)\n        ax2.axvline(0, color='black', linewidth=0.8)\n        for bar, val in zip(bars2, vs):\n            ax2.text(val + (0.002 if val >= 0 else -0.002),\n                     bar.get_y() + bar.get_height()/2,\n                     f'{val:+.3f}', va='center',\n                     ha='left' if val >= 0 else 'right', fontsize=9)\n        ax2.set_xlabel(\"SHAP Value (impact on predicted score)\")\n        ax2.set_title(\"Green bars = increased the score  |  Red bars = decreased the score\")\n        ax2.spines[['top','right']].set_visible(False)\n        plt.tight_layout(); st.pyplot(fig2); plt.close()\n\n        c1, c2 = st.columns(2)\n        with c1:\n            st.markdown(\"**Top factors that HELPED this score:**\")\n            for f, v in sorted(zip(fn, vs), key=lambda x: x[1], reverse=True):\n                if v > 0:\n                    st.markdown(f\"- **{f}** pushed score up by `{v:+.3f}`\")\n        with c2:\n            st.markdown(\"**Top factors that HURT this score:**\")\n            for f, v in sorted(zip(fn, vs), key=lambda x: x[1]):\n                if v < 0:\n                    st.markdown(f\"- **{f}** pushed score down by `{v:+.3f}`\")\n\n        st.divider()\n\n        # \u2500\u2500 Baseline comparison \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n        st.markdown(\"<div class='section-header'>Model Comparison \u2014 Our System vs Naive Baselines</div>\",\n                    unsafe_allow_html=True)\n        st.caption(\n            \"Comparison between our semantic matching system and two standard baselines. \"\n            \"Higher score = better match detection.\"\n        )\n        model_names  = ['Keyword Overlap\\n(Baseline 1)', 'TF-IDF + Cosine\\n(Baseline 2)', 'Our Model\\n(SBERT + XGBoost)']\n        model_scores = [result['baseline_keyword'], result['baseline_tfidf'], result['overall_score']]\n        bar_cols     = ['#adb5bd', '#6c757d', '#28a745' if result['overall_score'] >= 75\n                        else '#ffc107' if result['overall_score'] >= 50 else '#dc3545']\n\n        fig3, ax3 = plt.subplots(figsize=(9, 4))\n        bars3 = ax3.bar(model_names, model_scores, color=bar_cols, edgecolor='none', width=0.5)\n        ax3.set_ylim(0, 110)\n        ax3.set_ylabel(\"Match Score (%)\")\n        ax3.set_title(\"Why our model is better than keyword matching\")\n        for bar, val in zip(bars3, model_scores):\n            ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,\n                     f'{val}%', ha='center', fontsize=12, fontweight='bold')\n        ax3.spines[['top','right']].set_visible(False)\n        plt.tight_layout(); st.pyplot(fig3); plt.close()\n\n        bc1, bc2, bc3 = st.columns(3)\n        with bc1:\n            st.metric(\"Keyword Overlap\", f\"{result['baseline_keyword']}%\",\n                      help=\"Counts common words between resume and JD. Ignores meaning.\")\n        with bc2:\n            st.metric(\"TF-IDF + Cosine\", f\"{result['baseline_tfidf']}%\",\n                      help=\"Term frequency weighting with cosine similarity. No semantic understanding.\")\n        with bc3:\n            delta = round(result['overall_score'] - result['baseline_tfidf'], 1)\n            st.metric(\"Our Model (SBERT + XGBoost)\", f\"{result['overall_score']}%\",\n                      delta=f\"{delta:+}% vs TF-IDF\",\n                      help=\"Semantic embeddings + trained ML model + SHAP explainability.\")\n\n        st.divider()\n\n        # \u2500\u2500 Explanation + improvements \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n        c1, c2 = st.columns(2)\n        with c1:\n            st.markdown(\"<div class='section-header'>Explanation</div>\", unsafe_allow_html=True)\n            st.info(result['explanation'])\n        with c2:\n            st.markdown(\"<div class='section-header'>Resume Improvement Tips</div>\", unsafe_allow_html=True)\n            for i, tip in enumerate(result['improvements'], 1):\n                st.markdown(f\"**{i}.** {tip}\")\n\n        st.divider()\n\n        # \u2500\u2500 Updated resume \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n        st.markdown(\"<div class='section-header'>Updated Resume (ATS Optimized)</div>\",\n                    unsafe_allow_html=True)\n        st.caption(\n            \"This is your resume rewritten to better match the job description. \"\n            \"Missing critical skills have been added, and ATS notes show exactly what to improve.\"\n        )\n        st.text_area(\"Updated Resume\", value=result['updated_resume'], height=400)\n        st.download_button(\n            label=\"Download Updated Resume (.txt)\",\n            data=result['updated_resume'],\n            file_name=f\"updated_resume_{result['candidate'].replace(' ','_')}.txt\",\n            mime=\"text/plain\",\n            use_container_width=True\n        )\n\n        st.divider()\n\n        # \u2500\u2500 Download full report \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n        save = {k: v for k, v in result.items()\n                if k not in ['shap_vals', 'feature_cols', 'updated_resume']}\n        st.download_button(\n            label=\"Download Full Analysis Report (JSON)\",\n            data=json.dumps(save, indent=2, default=str),\n            file_name=f\"analysis_{result['candidate'].replace(' ','_')}.json\",\n            mime=\"application/json\",\n            use_container_width=True\n        )\n"

with open('/content/app.py', 'w') as f:
    f.write(app_code)
print('app.py written successfully!')

## Cell 3 — Launch dashboard

In [ ]:
import subprocess, threading, time
from pyngrok import ngrok

subprocess.run(['pkill', '-f', 'streamlit'], capture_output=True)
time.sleep(2)

def run_streamlit():
    subprocess.run([
        'streamlit', 'run', '/content/app.py',
        '--server.port=8501',
        '--server.headless=true',
        '--server.enableCORS=false',
        '--server.enableXsrfProtection=false'
    ])

thread = threading.Thread(target=run_streamlit, daemon=True)
thread.start()
time.sleep(6)

ngrok.set_auth_token('3C0yr1HtN9QbyVgQVknHbdrWaTB_7YgpDZPo4eLmUfb4SMuV3E')  # paste your ngrok token
public_url = ngrok.connect(8501)
print('=' * 55)
print('DASHBOARD IS LIVE!')
print('=' * 55)
print(f'Click this URL: {public_url}')
print('Keep this cell running while using the dashboard.')